In [1]:
!pip install transformers datasets accelerate sentencepiece pandas scikit-learn huggingface_hub

In [16]:
import pandas as pd
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

In [3]:
if torch.cuda.is_available():
    print("GPU (CUDA) is available and will be used for training.")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"Current GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU (CUDA) is not available. Training will proceed on CPU.")

# The TrainingArguments object (defined in a previous cell) by default will use CUDA if available.
# No explicit modification to training_args is needed for default GPU usage.

GPU (CUDA) is available and will be used for training.
Number of GPUs available: 1
Current GPU device name: Tesla T4


In [4]:
df = pd.read_csv("/content/Lables.csv")

df.head()

,Id,UserId,MessageContent,Time,Sender,MessageId,Type,Amount,Target,Alias,Account
0,b79bbf7c-6e8b-4356-929b-026c8277e390,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-26 11:44:18+00,AX-ICICIT-S,cc53375a-447d-4b42-a819-7e1c9f6db10b,Debit,80.00,AWFIS,Office Canteen,ICICI - XX789
1,ec6680c2-6aa9-4675-8ac2-7cd1f4820a76,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 306.21 on...,2025-12-26 15:24:11+00,AX-ICICIT-S,3cab480f-410f-4482-a685-afb1db248290,Debit,306.21,ZOMATO,NaN,ICICI - XX789
2,e9fb8ada-2666-401e-b7be-152c9a9c7fdd,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Credited for Rs:5000.00 on 26-12-202...,2025-12-26 15:41:35+00,JD-UNIONB-S,c82bf5ba-4931-4fd3-b711-ff40c1a36c1c,Credit,5000.00,NaN,NaN,UNIONB - *0186
3,80da7ab7-8b35-4132-8e33-11a22f124dbf,9c86307b-8354-4575-86e7-89258e57881e,A/c *0186 Debited for Rs:600.00 on 28-12-2025 ...,2025-12-28 15:01:24+00,JD-UNIONB-S,f645a1aa-386c-4872-a25b-569469cabcb7,Debit,5173.95,NaN,NaN,UNIONB - *0186
4,d290bf24-3959-4cb8-95ab-2b1309b3955c,9c86307b-8354-4575-86e7-89258e57881e,ICICI Bank Acct XX789 debited for Rs 80.00 on ...,2025-12-28 15:21:45+00,AX-ICICIT-S,58f58044-fe67-44f6-9465-23d9938727c4,Debit,80.00,MRS GEDDA RAMA,NaN,ICICI - XX789


In [23]:
def preprocess(example):
    model_input = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    labels = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    model_input["labels"] = labels["input_ids"]

    return model_input
def build_input(row):
    return f"extract transaction details: Sender: {row['Sender']} Message: {row['MessageContent']}"

def build_output(row):
    output = {
        "type": row["Type"],
        "amount": row["Amount"],
        "target": row["Target"],
        "alias": row["Alias"],
        "account": row["Account"]
    }
    return json.dumps(output)

In [24]:
df["input"] = df.apply(build_input, axis=1)
df["output"] = df.apply(build_output, axis=1)

df = df[["input", "output"]]
df.head()

KeyError: 'Sender'

In [25]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [26]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [27]:
train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [17]:
training_args = TrainingArguments(
    output_dir="./transaction_model",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,  # Increased from 3 to 10 epochs
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    push_to_hub=False
)

In [29]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator # Added data collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,10.431043
2,No log,10.360575
3,No log,10.290594
4,No log,10.222397
5,No log,10.157288
6,No log,10.098988
7,No log,10.051723
8,No log,10.015087
9,No log,9.990150
10,No log,9.978760


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=10.299606323242188, metrics={'train_runtime': 443.0354, 'train_samples_per_second': 0.339, 'train_steps_per_second': 0.045, 'total_flos': 51356801433600.0, 'train_loss': 10.299606323242188, 'epoch': 10.0})

In [30]:
def predict(message, sender):

    text = f"Sender: {sender} Message: {message}"

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(**inputs, max_length=128)

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

In [31]:
def predict(message, sender):
    text = f"extract transaction details: Sender: {sender} Message: {message}"
    inputs = tokenizer(text, return_tensors="pt")
    # Move inputs to the model's device (GPU in this case)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=128)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

predict(
    "ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; AWFIS credited",
    "AX-ICICIT-S"
)

'ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; AWFIS credited'

In [43]:
from huggingface_hub import notebook_login

notebook_login()

In [44]:
trainer.push_to_hub("sms-transaction-extractor")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_model/model.safetensors:   1%|          | 8.31MB /  990MB            

  ...n_model/training_args.bin:   8%|7         |   390B / 5.14kB            

CommitInfo(commit_url='https://huggingface.co/srisaidivyakola/transaction_model/commit/dc5440b639afbb672c8138413a138463fa7737d0', commit_message='sms-transaction-extractor', commit_description='', oid='dc5440b639afbb672c8138413a138463fa7737d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/srisaidivyakola/transaction_model', endpoint='https://huggingface.co', repo_type='model', repo_id='srisaidivyakola/transaction_model'), pr_revision=None, pr_num=None)

In [41]:
from huggingface_hub import logout
logout()
print("Logged out from Hugging Face Hub.")

Not logged in!


Logged out from Hugging Face Hub.
